<a href="https://colab.research.google.com/github/juleuzun/Python_DataScience_Bootcamp/blob/main/Lessons/Lesson_08_RNN_Recurrent_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Recurrent Neural Network

# Tokenizer

In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
data = [
    'This is an example sentence.',
    'There is another sentence as well.',
    'A different example text.'
]

In [3]:
tokenizer = Tokenizer() # oov_token="<OOV>"
tokenizer.fit_on_texts(data)
numerical_sequences = tokenizer.texts_to_sequences(data)

In [4]:
print("word_index: ",tokenizer.word_index)


word_index:  {'is': 1, 'example': 2, 'sentence': 3, 'this': 4, 'an': 5, 'there': 6, 'another': 7, 'as': 8, 'well': 9, 'a': 10, 'different': 11, 'text': 12}


In [5]:
for number in numerical_sequences:
  print(number)

[4, 1, 5, 2, 3]
[6, 1, 7, 3, 8, 9]
[10, 11, 2, 12]


In [6]:
padded_sequences = pad_sequences(numerical_sequences, maxlen=5)

In [7]:
for number in padded_sequences:
  print(number)

[4 1 5 2 3]
[1 7 3 8 9]
[ 0 10 11  2 12]


# Embedding

In [8]:
from keras.layers import Embedding
import numpy as np

In [9]:
vocab_size = len(tokenizer.word_index) + 1

embedding_dim = 300

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=5
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
embedded_sequences = embedding_layer(np.array(padded_sequences))

In [11]:
print("Embedding Layer Output: ")
print(embedded_sequences)

Embedding Layer Output: 
tf.Tensor(
[[[ 0.01769102  0.02104254  0.03310708 ... -0.03842659  0.01384724
    0.00495543]
  [ 0.00800083  0.00740501 -0.00224447 ...  0.00026445  0.03649311
   -0.01721771]
  [-0.0257417   0.01284863 -0.02436307 ... -0.02468394 -0.0020945
   -0.00579542]
  [ 0.03413541  0.00101286 -0.02480841 ...  0.01012602  0.00253179
    0.02909236]
  [ 0.02521021 -0.04452863 -0.04089329 ...  0.02844331 -0.03825856
    0.01907079]]

 [[ 0.00800083  0.00740501 -0.00224447 ...  0.00026445  0.03649311
   -0.01721771]
  [-0.0068421  -0.01357083 -0.00661744 ...  0.0251424   0.01381757
   -0.04156125]
  [ 0.02521021 -0.04452863 -0.04089329 ...  0.02844331 -0.03825856
    0.01907079]
  [ 0.04738389 -0.010172   -0.02387651 ...  0.04907674  0.02441508
   -0.03494577]
  [ 0.01201689 -0.04911381  0.00482869 ...  0.02212871  0.03020299
    0.03899534]]

 [[-0.00265963  0.03908798 -0.01506307 ...  0.00860615 -0.00839101
    0.02265489]
  [ 0.04456604  0.03575701  0.0164029  ...  0.00

# RNN

In [12]:
!pip install joblib
!pip install colorama

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from keras.layers import SimpleRNN, Dense
from keras.utils import to_categorical
import joblib
from colorama import Fore, Back, Style
from google.colab import drive

In [14]:
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
data_path = '/content/drive/MyDrive/TechIstanbul/Phishing_Email.csv'
model_path = '/content/drive/MyDrive/TechIstanbul/rnn.h5'

In [16]:
mail_data = pd.read_csv(data_path)
mail_data = mail_data.where((pd.notnull(mail_data)),'')

In [17]:
mail_data.loc[mail_data['Email Type']=='Safe Email','Email Type'] = 0
mail_data.loc[mail_data['Email Type']=='Phishing Email','Email Type'] = 1

In [18]:
X = mail_data['Email Text']
y = mail_data['Email Type']

In [19]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
indexed_data = tokenizer.texts_to_sequences(X)

In [20]:
word_index = tokenizer.word_index
word_length = 1000
fixed_length = pad_sequences(indexed_data, maxlen=word_length)

In [21]:
y = to_categorical(y)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(fixed_length, y, test_size=0.2, random_state=42)

In [23]:
model = Sequential()
model.add(Embedding(len(word_index)+1,output_dim=128, input_length=word_length))
model.add(SimpleRNN(64,activation='relu'))
model.add(Dense(2,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model.fit(X_train,y_train, epochs=5, batch_size=32, validation_data=(X_test,y_test))

Epoch 1/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 297s 630ms/step - accuracy: 0.8332 - loss: 5332.7739 - val_accuracy: 0.9523 - val_loss: 0.1377
Epoch 2/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 291s 623ms/step - accuracy: 0.9725 - loss: 0.0715 - val_accuracy: 0.9638 - val_loss: 0.0974
Epoch 3/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 295s 632ms/step - accuracy: 0.9855 - loss: 0.0375 - val_accuracy: 0.9643 - val_loss: 0.1068
Epoch 4/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 321s 631ms/step - accuracy: 0.9873 - loss: 0.0290 - val_accuracy: 0.9496 - val_loss: 0.1499
Epoch 5/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 291s 624ms/step - accuracy: 0.9879 - loss: 0.0264 - val_accuracy: 0.9598 - val_loss: 0.1272


In [25]:
model.save(model_path)

In [26]:
import tensorflow as tf
model = tf.keras.models.load_model(model_path, compile=False)
#When performing training, remove the compile parameter and run the following.
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
while True:
    user_input = input('Enter the email message: ')
    tokenized_word = tokenizer.texts_to_sequences([user_input])
    fixed_length = pad_sequences(tokenized_word, maxlen=word_length)
    prediction = model.predict(fixed_length)

    if np.argmax(prediction) == 1:
        print(Fore.RED, 'This might be a phishing email.')
    else:
        print(Fore.GREEN, 'This is a safe email.')

    print(Fore.BLUE, 'Accuracy score: ', prediction[0][np.argmax(prediction)])

Enter the email message: make her beg you to give it to her everynight you too can please you and your partner so much better . and it doesn ' t take half a year like other solutions . if you don ' t trust me , trust the new york times , newsweek , time magazine , and the testemonials of many people all over the net . look , if anything else check out the site and see the facts for yourself its new , its safe ! its the most advanced penile enlargement solution ! it ' s 100 % guaranteed to enlarge your penis . 3 + inches magna - rx patch - click here the amazing , new magna rx patch is not available in any stores or on other websites . accept no imitations ! order your male enhancement patch now through this exclusive website offer and get a 1 - month supply free ! one small investment in yourself will last a lifetime ! - no pills or capsules - no lotions or cremes - no pumps , weights , or exercises - no prescription necessary - doctor designed endorsed - 100 % safe natural - more stam